In [ ]:
import uproot
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

file_name = '../../MCEventInfoForBinningOptimisation.root'

with uproot.open(file_name) as f:
    # Load some data from the file
    df_events = f['MCEvent'].arrays([
                                    'TrueENu',
                                    'TrueCZ',
                                    'TrueEHad',
                                    'TrueELep',
                                    'RecoENu',
                                    'RecoEHad',
                                    'RecoELep',
                                    'RecoCZ',
                                    'Sample',
                                    'EventWeight'
                                    ],
                                     library='pd')

df_events['RecoY'] = df_events['RecoEHad'] / df_events['RecoENu']

samples = {
            'FC_numu': 0,
            'FC_nue': 1,
            'PC_numu': 2,
            'PC_nue': 3,
            }

display(df_events)

In [ ]:
def create_energy_bins(cp_step, m32_step, bsm_step):

    delta_cp_bins = np.arange(0., 1., cp_step)
    mass_ordering_bins = np.arange(1, 10, m32_step)
    bsm_bins = np.arange(10, 100+bsm_step, bsm_step)

    E_nu_bins = np.concatenate([delta_cp_bins,mass_ordering_bins,bsm_bins])
    return E_nu_bins

E_nu_bins = create_energy_bins(0.2, 0.05, 0.5)
#print(f"{len(E_nu_bins)}: {E_nu_bins}")

In [ ]:
class SampleBinOptimiser:
    def __init__(self, name, df, initial_bins, axes_factors={}, min_mc=100, min_data=3., exposure=400., nominal_exposure=400.):
        
        self.name=name
    
        self.df = df.reset_index(drop=True)

        self.initial_bins=initial_bins

        self.axes=list(self.initial_bins.keys())

        self.optimised_bins=dict.fromkeys(self.initial_bins.keys())
        self.axes_factors=dict.fromkeys(self.initial_bins.keys())

        for axis, edges in self.initial_bins.items():
            # add overflow bin to reproduce numpy histogram behaviour
            # TODO check if inf is already present
            self.optimised_bins[axis] = np.concatenate([edges, [float('inf')]])
            self.df[axis+'binIndex'] = np.nan

            axis_factor=1.
            if axis in axes_factors:
                axis_factor=axes_factors[axis]
            self.axes_factors[axis]=axis_factor

        # constants for algo.
        self.nominal_exposure=nominal_exposure
        self.exposure=exposure
        self.expsure_scaling=exposure/nominal_exposure

        self.min_mc=min_mc
        self.min_data=min_data

    ###
    def optimise_bins(self):

        for i, axis in enumerate(self.axes):

            bins_array=list(self.optimised_bins.values())
            target_bins=bins_array[i]

            #print(f"optimising {axis} {target_bins}")

            outer_axes=self.axes[:i+1]
            outer_axes.pop()

            outer_bins=bins_array[:i+1]
            outer_bins.pop()

            outer_n_bins = [len(x)-2 for x in outer_bins]

            #print(f"outer {outer_axes} {outer_n_bins} {outer_bins}")

            self._optimise_bins_loop(outer_n_bins, outer_axes, axis)
        return self.optimised_bins
    ###

    ###
    def _optimise_bins_loop(self, outer_bins, outer_axes, axis, indices=()):
        if not outer_bins:

            true_array = [True] * len(self.df)

            bin_mask=pd.Series(true_array, index=self.df.index)
            bin_indices =  [idx+1 for idx in indices]
            for i, bin_idx in enumerate(bin_indices):
                o_a=outer_axes[i]
                bin_mask=bin_mask & (self.df[o_a+'binIndex'] == bin_idx)
                #print(f"masking {o_a} bin {bin_idx}")

            sel_idx = self.df.index[ bin_mask ]
            outer_axes_str=f"{outer_axes} bin {bin_indices}"
            print(f"checking {outer_axes_str} if {axis} bins are ok  with total: {len(sel_idx)}")

            ok = self._optimise_bins_lopp_work(outer_axes_str, axis, sel_idx)

            return
        for i in range(outer_bins[0]):
            self._optimise_bins_loop(outer_bins[1:], outer_axes, axis, indices + (i,))
    ###

    ###
    def _optimise_bins_lopp_work(self, outer_axes_str, axis, idx_sel):
    
        while True:

            df_sel=self.df.iloc[idx_sel]
            data=df_sel[axis].to_numpy()
            weights=df_sel['EventWeight'].to_numpy()

            ok, indices, new_edges = SampleBinOptimiser.optimise_edges(axis,
                                                    data,
                                                    weights, 
                                                    self.optimised_bins[axis],
                                                    self.expsure_scaling,
                                                    self.min_mc,
                                                    self.min_data*self.axes_factors[axis]
                                                    )
            if ok:
                print(f"{outer_axes_str} {axis} ok")
                self.df.iloc[idx_sel, self.df.columns.get_loc(axis+'binIndex')]=indices
                break
            else:
                print(f"{outer_axes_str} {axis} were optimised")
                self.optimised_bins[axis]=new_edges

        return True
    ###

    ###
    @staticmethod
    def optimise_edges(axis, data, weights, edges, data_scaling, min_mc, min_data):

        new_edges = edges.copy()

        indices, mc_counts, data_counts = SampleBinOptimiser.get_bins_and_counts(axis, data, weights, edges, data_scaling)

        n_bins=len(mc_counts[:-1])

        ok=False
        bad_edges=[]
        bin_idx = 0
        while bin_idx < n_bins:
            mc_count=mc_counts[bin_idx]
            data_count=data_counts[bin_idx]
            while True:
                if mc_count < min_mc or data_count < min_data:
                    #print(f"bad bin: {bin_idx} {mc_count}, {data_count}")

                    # last bin, special
                    if bin_idx == n_bins-1:
                        # actually need to merge with last good bin
                        if len(new_edges) < 4:
                            raise RuntimeError(f"all bins are bad! data sum {np.sum(data_counts[:-1])} mc sum {np.sum(mc_counts[:-1])}")
                        else:
                            bad_edge=new_edges[::-1][2]
                            new_edges=np.delete(new_edges, np.where(new_edges == bad_edge))
                            bad_edges.append(bad_edge)
                            bin_idx += 1
                            break
                    else:
                        bin_idx += 1
                        bad_edge=edges[bin_idx]
                        new_edges=np.delete(new_edges, np.where(new_edges == bad_edge))
                        bad_edges.append(bad_edge)
                        mc_count=mc_count+mc_counts[bin_idx]
                        data_count=data_count+data_counts[bin_idx]
                        #print(f"merged {mc_count}, {data_count}")
                else:
                    bin_idx += 1
                    break

        data_sum=np.sum(data_counts[:-1])
        mc_sum=np.sum(mc_counts[:-1])
        print(f"edges: {edges[:-1]}")
        print(f"data counts: {data_counts[:-1]}, data sum: {data_sum}")
        print(f"mc counts: {mc_counts[:-1]}, mc sum:  {mc_sum}")

        if (len(new_edges)==len(edges)):
            ok=True
        else:
            print(f"removed edges: {bad_edges}")
            print(f"new {axis} edges: {new_edges}")
            if (len(new_edges)==3):
                #print(f"[WARNING] axis {axis} fully merged! data sum {data_sum} mc sum {mc_sum}")
                raise RuntimeError(f"[ERROR] axis {axis} fully merged!")
                
        return ok, indices, new_edges
    ###

    ###
    @staticmethod
    def get_bins_and_counts(axis, data, weights, edges, data_scaling):
        mc_counts=[]
        data_counts=[]
    
        indices = np.digitize(data, edges, right=False)
    
        for bin in range(1,len(edges)):
            mc_c = np.count_nonzero(indices == bin)
            mc_counts.append(mc_c)
    
            bin_weigths=weights[indices == bin]
            data_c=np.sum(bin_weigths)*data_scaling
            data_counts.append(data_c)
        return indices, mc_counts, data_counts
    ###

optimised_sample_bins=dict.fromkeys(samples.keys())

# order of axes here determines optimisation order
histo_axes_bins_initial = {
                        'RecoCZ': np.arange(-1., 1.01, 0.2),
                        'RecoENu': E_nu_bins,
                        'RecoY': np.arange(0., 1.01, 0.1),
                        }
exppsure=400.

for sample, sample_id in samples.items():
    
    sample_bin_opt=SampleBinOptimiser(sample, 
                                      df_events[df_events['Sample']==sample_id], 
                                      histo_axes_bins_initial,
                                      {'RecoENu': 10.},
                                      exposure=exppsure
                                      )
    print(f"{sample}")
    optimised_sample_bins[sample]=sample_bin_opt.optimise_bins()
    print("")

print(f"exposure {exppsure} kt-y")

for sample in samples.keys():
    for axis, edges in optimised_sample_bins[sample].items():
        print(f"Final {sample} {axis} edges: {edges[:-1]}")